# Weather Hazard Category · Random Forest Classifier

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts WHC exactly +7 days ahead*

This notebook trains and evaluates the **Random Forest Classifier** model for the **Weather Hazard Category (WHC)**, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

A Random Forest with class weighting will achieve **higher validation macro F1 than all three
baselines**, because averaging decorrelated trees gives stable class estimates even when the
minority classes are small.

The competing possibility: with so few minority-class days, most trees barely see them in their
bootstrap sample and the vote collapses onto the majority class. The confusion matrix in section 8
detects this directly.

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `RandomForestClassifier` | Low-variance ensemble with native class weighting |
| Imbalance | `class_weight` over none, balanced, balanced_subsample | `balanced_subsample` recomputes weights for each bootstrap sample |
| Scaling | None | Trees split on thresholds, so scale is irrelevant |
| Strategy | One classifier for the single +7 day horizon | Matches the deployed artefact layout |
| Selection | Optuna TPE sampler, 25 trials per target | See section 6 |
| Objective | validation **macro F1** (higher is better) | See section 6 |

### Workflow

All ten experiment notebooks follow the same steps and run the same code from
`nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model
specification in section 1 differs between notebooks.**

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data | Training and validation splits (the test split is never loaded here) |
| 3 | Baselines | The floor every model must clear |
| 4 | Default configuration | An untuned reference point |
| 5 | Feature-family ablation | What each block of engineered features contributes |
| 6 | Hyperparameter selection | Choose settings on the validation split |
| 7 | Seed stability | Is the tuning gain real or noise? |
| 8 | Final model | Tuned model against the default and the baselines |
| 9 | Feature importance | Which features the model relies on |
| 10 | Generalisation check | Training vs validation performance |
| 11 | Save artefacts | Model bundle + `/model-metadata` document |
| 12 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the
project package, so no helper is redefined in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "weather_hazard"
EXPERIMENT = "whc_random_forest"
ALGORITHM = "Random Forest Classifier"
N_TRIALS = 25  # Optuna trials per target (None when a fixed grid is used)

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It defines how the estimator is built, a sensible
default configuration, a cheaper configuration for the ablation, and the candidate settings for
section 6 (either an Optuna search space or a fixed grid).

`min_samples_leaf` needs care here: if every leaf must hold more rows than a rare class contains,
that class becomes unreachable. `max_features` stays low-capable because of the near-duplicate
column blocks.

In [ ]:
def build_estimator(params: dict, seed: int) -> RandomForestClassifier:
    """A random forest classifier."""
    return RandomForestClassifier(n_jobs=-1, random_state=seed, **params)


DEFAULT_PARAMS = {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 1,
                  "min_samples_split": 2, "max_features": 0.3, "class_weight": "balanced"}
ABLATION_PARAMS = {**DEFAULT_PARAMS, "n_estimators": 150}


def search_space(trial) -> dict:
    """Forest size, tree complexity, feature subsampling and class weighting."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),
        "max_depth": trial.suggest_categorical("max_depth", [None, 6, 10, 16, 24]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "max_features": trial.suggest_float("max_features", 0.05, 1.0),
        "class_weight": trial.suggest_categorical(
            "class_weight", [None, "balanced", "balanced_subsample"]),
    }


SEARCH_SPACE, GRID = search_space, None

In [ ]:
exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    build=build_estimator,
    default_params=DEFAULT_PARAMS,
    ablation_params=ABLATION_PARAMS,
    search_space=SEARCH_SPACE,
    grid=GRID,
)

---
## 2. Data

Only the **training** and **validation** splits are loaded. The test split is reserved for a
single evaluation in the comparison notebook, once one model has been selected, so nothing in
this notebook can be influenced by it.

> The preprocessing notebook already applied the chronological split, the warm-up filter and an
> embargo at each boundary. Nothing is re-split or shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Three naive forecasts, all fitted on the training split only:

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Majority class** | Always predict the most common class | Nothing but the class balance |
| **Persistence** | The class in 7 days equals today's class (from today's WHI) | The current weather |
| **Seasonal mode** | The most common class for that month | The season |

The majority-class rule scores a high *accuracy* but a low *macro F1*, which is exactly why
accuracy is not used to select models for this imbalanced target.

In [ ]:
exp.run_baselines()

---
## 4. Default configuration

The untuned default is scored before any search. It is the reference for judging what tuning
bought in section 8; a search that fails to beat a sensible default is a result worth reporting.
`best_baseline` is repeated for comparison.

In [ ]:
exp.run_defaults()

---
## 5. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone (a single column),
and the model is refitted at each step. This measures what each block of engineering contributes
instead of assuming it. The cheaper `ABLATION_PARAMS` are used throughout so the differences come
from the features, not from tuning. Scores are on validation (macro F1).

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 6. Hyperparameter selection

The objective is validation **macro F1** (higher is better).

**Why macro F1.** It averages F1 across classes without weighting by frequency, so a model that
ignores the rare hazard classes cannot hide behind the common Low Risk days. Accuracy would reward
exactly that behaviour. The average is taken over the classes present in training, because a
class the model never saw cannot be predicted.

**Why validation, not cross-validation.** The split is chronological. Shuffled k-fold would put
future days into training and inflate every score, because consecutive days are strongly
correlated. The fixed 2024 validation year matches how the model will be used: trained on the
past, applied forward.

> **Stated plainly:** selecting on validation makes the validation score slightly optimistic. The
> honest estimate comes from the test split, which only the comparison notebook reads, once.

In [ ]:
exp.tune(n_trials=N_TRIALS)

In [ ]:
exp.plot_search()
plt.show()

---
## 7. Seed stability

The selected configuration is refitted with three different random seeds. If the score moves by
as much as the spread between the top search candidates (`search_beats_noise = False`), the search
was ranking noise and the conclusion should say so. Deterministic models show a spread of zero.

In [ ]:
exp.seed_stability()

---
## 8. Final model against the baselines

The selected configuration is refitted and compared with the default configuration and every
baseline. `gain_of_tuned_model` is positive when the tuned model is better than that row.

Read the **confusion matrix** as well as the headline score: a model that almost never predicts
the rare classes has not learned to flag hazardous days, whatever its macro F1. The printed
boundary summary shows how many errors sit within a few WHI points of a class threshold; those are
threshold effects rather than genuine misses.

In [ ]:
exp.finalise()

In [ ]:
exp.plot_validation()
plt.show()

---
## 9. Feature importance

Tree ensembles report **impurity-based importance** directly from the fitted model. Comparing horizons is informative: features that rise in rank from *t*+1 to *t*+3 carry genuine forecast signal rather than persistence.

> Correlated features (a variable, its lags and its rolling means) share importance, so a feature
> that looks unimportant on its own may belong to an essential block. The ablation in section 5 is
> the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 10. Generalisation check

Training and validation scores side by side. A small gap means the model generalises; a large
gap means it has memorised patterns in the training years that do not carry over to 2024. Read
it together with the seed-stability table in section 7.

In [ ]:
exp.generalisation()

---
## 11. Save artefacts

The fitted model(s) are written to `models/weather_hazard/experiments/whc_random_forest/` with:

| File | Contents |
|:--|:--|
| `whc_random_forest_h<horizon>.joblib` | One fitted model per horizon |
| `metadata.json` | Target, algorithm, horizon, features, importance, hyperparameters, metrics — the `/model-metadata` document |
| `manifest.json` | File list and the feature column order the API must use |

> The feature column **order** is recorded because models receive an unnamed array at prediction
> time and would silently give wrong answers if the columns were permuted.

In [ ]:
exp.export()

---
## 12. Conclusions

- **Did the forest ever predict a rare class?** Check the `predicted` column of the per-class
  table. If the rare classes are never predicted, macro F1 is carried by the common classes and the
  model cannot flag hazardous days.
- **Which weighting won.** If `balanced_subsample` beats `balanced`, minority-class scarcity
  inside individual trees was the binding constraint.
- **Against boosting.** A forest that matches XGBoost while being simpler is a legitimate
  deployment recommendation, especially given how noisy macro F1 is on this validation period.

> The test split has still not been used. It is scored exactly once, in the
> `whc_comparison` notebook, after one model has been selected on validation
> performance across all five WHC experiments.